# Week 4 in Agentic course exercise

## Daily News Update Solution using langGraph and agents

#### This solution allows the user to use voice input or type by hand on the textarea.
#### The user ask the solution for news update for the day, the system searches the web using tavily tool, the pass the result to a summarizer agent who summarizes the output from the researcher.
#### The summarizer hands over to formatter and then to the emailer which sends the news update the user email if the option is selected,

## Key components/Tools used:
1. Tavily for web research tool
2. Openai "gpt-4o-mini" for summarizer
3. user_email address
4. user email password
4. Gradio UI
5. langGraph (Agentic   Workflow)

## Agents/Tools
1. Researcher (Tool)
2. Summarizer (Agent)
3. Formatter  (Tool)
4. Query builder (Agent)



## Import libs

In [ ]:
import argparse
import logging
import os
import smtplib
from email.mime.text import MIMEText
from typing import TypedDict

import gradio as gr
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.graph import END, START, StateGraph
from openai import OpenAI

In [ ]:
# Load environment variables
load_dotenv(override=True)


## Define State

In [ ]:

class State(TypedDict, total=False):
    """Shared LangGraph state (each field is a graph channel)."""

    user_request: str
    send_email: bool
    raw_news: str
    processed_news: str
    email_body: str

 # Set Default QUERIES

DEFAULT_QUERIES = [
    "Latest stock market news",
    "Latest political news",
    "Latest business and entrepreneur news",
    "Latest innovations and technology news",
]


In [ ]:
# Initilize the llm and tavily search tool

search_tool = TavilySearch()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Add audio transcriber function
def transcribe_audio(audio_path: str | None) -> str:
    if not audio_path or not os.path.isfile(audio_path):
        return ""
    client = OpenAI()
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1", file=audio_file
        )
    return (transcript.text or "").strip()


# Agents and Tools Nodes

In [ ]:
# Audio Transcriber function
def transcribe_audio(audio_path: str | None) -> str:
    if not audio_path or not os.path.isfile(audio_path):
        return ""
    client = OpenAI()
    with open(audio_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            model="whisper-1", file=audio_file
        )
    return (transcript.text or "").strip()




In [ ]:
def build_search_queries(user_request: str | None) -> list[str]:
    text = (user_request or "").strip()
    if not text:
        return list(DEFAULT_QUERIES)
    prompt = (
        "The user described what news they want (may come from speech transcription).\n"
        f"USER REQUEST:\n{text}\n\n"
        "Reply with exactly 4 short web search queries (one per line, no numbering or bullets). "
        "Cover their interests and sensible related angles for current news."
    )
    try:
        response = llm.invoke(prompt)
        lines = [
            ln.strip().lstrip("-•0123456789.) ").strip()
            for ln in response.content.strip().splitlines()
            if ln.strip()
        ]
        queries = [ln for ln in lines if ln][:4]
        for i, fallback in enumerate(DEFAULT_QUERIES):
            if len(queries) > i:
                continue
            queries.append(fallback)
        return queries[:4]
    except Exception as e:
        logging.warning("LLM query expansion failed (%s); blending user line with defaults.", e)
        return [text] + list(DEFAULT_QUERIES[:3])

In [ ]:
# Researcher function
def researcher(state: State) -> State:
    logging.info("researcher received state: %r", state)
    logging.info("Starting research phase.")
    queries = build_search_queries(state.get("user_request"))

    results = []
    for q in queries:
        try:
            logging.info("Querying: %s", q)
            result = search_tool.run(q)
            results.append(result)
        except Exception as e:
            logging.error("Error during search for '%s': %s", q, e)
            results.append(f"Error fetching news for: {q}")

    results_str = [str(item) for item in results]
    state["raw_news"] = "\n\n".join(results_str)
    logging.info("Research phase complete.")
    return state



In [ ]:
# Agent
def summarizer(state: State) -> State:
    logging.info("summarizer received state: %r", state)
    logging.info("Starting summarization phase.")
    if not state.get("raw_news"):
        logging.error("No 'raw_news' found in state. State: %s", state)
        state["processed_news"] = "No news data available for summarization."
        return state

    focus = (state.get("user_request") or "").strip()
    focus_block = ""
    if focus:
        focus_block = (
            f"\nPrioritize angles that match what the user asked for:\n{focus}\n"
        )

    prompt = f"""You are a news analyst.{focus_block}
From the news below:
1. Extract stories
2. Categorize into:
   - Stocks
   - Politics
   - Business
   - Innovation and technology
3. Create:
   - Catchy headline
   - 2-3 line summary
Return in clean structured markdown.

NEWS:
{state["raw_news"]}
"""
    try:
        response = llm.invoke(prompt)
        state["processed_news"] = response.content
        logging.info("Summarization complete.")
    except Exception as e:
        logging.error("Error during summarization: %s", e)
        state["processed_news"] = "Error during summarization."
    return state

In [ ]:
# Formatter tool
def formatter(state: State) -> State:
    logging.info("formatter received state: %r", state)
    logging.info("Starting formatting phase.")
    content = state.get("processed_news", "No news to format.")
    html = f"""
    <html>
    <body>
    <h2>Daily News Update</h2>
    <div style='font-family: Arial; line-height:1.6;'>
    {content.replace("\n", "<br>")}
    </div>
    </body>
    </html>
    """
    state["email_body"] = html
    logging.info("Formatting complete.")
    return state

In [ ]:
# Emailer tool
def emailer(state: State) -> State:
    logging.info("emailer received state: %r", state)
    if state.get("send_email") is False:
        logging.info("Skipping email (preview / UI mode).")
        return state

    logging.info("Starting email phase.")
    msg = MIMEText(state.get("email_body", "No content."), "html")
    sender = os.getenv("EMAIL")
    recipient = os.getenv("EMAIL")
    msg["Subject"] = "Daily News Update"
    msg["From"] = sender
    msg["To"] = recipient
    logging.info("Email content to be sent:\n%s", state.get("email_body", "No content."))
    try:
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(sender, os.getenv("PASSWORD"))
            server.send_message(msg)
        logging.info("Email sent!")
    except Exception as e:
        logging.error("Error sending email: %s", e)
    return state

# Graph bullder

In [ ]:
graph_builder = StateGraph(State)

# Graph Nodes and Edges

In [ ]:
graph_builder.add_node("research", researcher)
graph_builder.add_node("summarizer", summarizer)
graph_builder.add_node("formatter", formatter)
graph_builder.add_node("emailer", emailer)

graph_builder.add_edge(START, "research")
graph_builder.add_edge("research", "summarizer")
graph_builder.add_edge("summarizer", "formatter")
graph_builder.add_edge("formatter", "emailer")
graph_builder.add_edge("emailer", END)



## UI implementation

In [ ]:

def launch_ui(server_name: str = "127.0.0.1", server_port: int = 7860) -> None:
    graph = graph_builder.compile()

    def process(
        audio_path: str | None,
        text_fallback: str,
        send_mail: bool,
    ) -> tuple[str, str, str, str]:
        transcript = ""
        try:
            if audio_path:
                transcript = transcribe_audio(audio_path)
            user_request = transcript.strip() or (text_fallback or "").strip()
            if not user_request:
                return "", "", "", "Provide a microphone recording and/or type a request."

            result = graph.invoke(
                {"user_request": user_request, "send_email": send_mail}
            )
            summary = result.get("processed_news") or ""
            html_body = result.get("email_body") or ""
            mail_note = (
                "Email sent (if SMTP credentials are valid)."
                if send_mail
                else "Email not sent (preview only)."
            )
            return transcript or user_request, summary, html_body, mail_note
        except Exception as e:
            logging.exception("UI workflow failed")
            return transcript, "", "", f"Error: {e}"

    with gr.Blocks(title="Daily news agent") as demo:
        gr.Markdown(
            "## Daily news agent\n"
            "Record **voice** (microphone) and/or type what news you want. "
            "Speech is transcribed with OpenAI **Whisper**; searches use **Tavily**. "
            "Uncheck **Send email** to only preview the digest in the browser."
        )
        audio = gr.Audio(
            sources=["microphone"],
            type="filepath",
            label="Microphone",
        )
        text_fallback = gr.Textbox(
            label="Or type your request",
            placeholder="e.g. AI startups in Europe and climate tech",
            lines=3,
        )
        send_mail = gr.Checkbox(label="Send email when done", value=False)
        run_btn = gr.Button("Run pipeline", variant="primary")

        transcript_out = gr.Textbox(label="Transcript / request used")
        status = gr.Textbox(label="Status")
        summary_out = gr.Markdown(label="Summary")
        html_preview = gr.HTML(label="Email HTML preview")

        run_btn.click(
            process,
            inputs=[audio, text_fallback, send_mail],
            outputs=[transcript_out, summary_out, html_preview, status],
        )

    demo.launch(server_name=server_name, server_port=server_port)




In [ ]:
# Launch the UI by default,
parser = argparse.ArgumentParser(description="Daily news LangGraph agent")
parser.add_argument(
    "--ui",
    action="store_true",
    help="Launch Gradio UI (voice + optional text)",
)
parser.add_argument(
    "--host",
    default="127.0.0.1",
    help="Bind address for Gradio (use 0.0.0.0 for LAN)",
)
parser.add_argument("--port", type=int, default=7860, help="Gradio server port")
args = parser.parse_args()

if args.ui:
    launch_ui(server_name=args.host, server_port=args.port)
